All Implementations are based on the paper **Collapsed Variational Bayes Inference of Infinite Relational Model** [See PDF](https://arxiv.org/abs/1409.4757)

# Import Libraries and Modules

In [ ]:
import numpy as np  # Used for matrices

# **Defining the Collapsed Gibbs Sampling (single-domain IRM)**

## 1. Counting Statistics


The authors define these counting statistics, which are maintained during sampling:

$$n_{k,l} = \sum_i \sum_j z_{1,i,k} z_{2,j,l} x_{i,j} \ , \quad N_{k,l} = \sum_i \sum_j z_{1,i,k} z_{2,j,l} (1 - x_{i,j}) \ , \tag{13}$$

Where $n_{k,l}$, and $N_{k,l}$ denote the number of positive and negative relation observations in the ($k,l$)-cluster pairs. And

$$m_{1,k} = \sum_i z_{1,i,k} \ , \quad m_{2,l} = \sum_j z_{2,j,l} \ . \tag{14}$$


Which are the number of nodes assigned to each cluster in each domain.


However, since we are building the single-domain model, we do not have a $Z_1$ and a $Z_2$. We just have our single list of assignments, $Z$, where $z_i$ represents the cluster assignment for node $i$, and $z_j$ represents the cluster assignment for node $j$.


In [ ]:
# 1. Initialize empty tally containers based on our random starting K
m_counts = np.zeros(K)
n_counts = np.zeros((K, K))
N_counts = np.zeros((K, K))

# 2. Populate the tallies
for i in range(N):
    m_counts[Z[i]] += 1  # Tally the nodes in each cluster
    for j in range(N):
        if X[i, j] == 1:
            n_counts[Z[i], Z[j]] += 1  # Tally the links
        else:
            N_counts[Z[i], Z[j]] += 1  # Tally the non-links


## 2. Gibbs Sampling Loop

We need to define the follwoing parts to perform the Gibbs Sampling Loop


*   **Remove node $i$'s influence from the tallies**
*   **Calculate the new probabilities**





### 1. **Remove node $i$'s influence from the tallies**


In the context of our code, because we already mathematically removed node $i$ from m_counts, m_counts[k] represents that exact $m_k^{-i}$ value, so because that denominator ($N - 1 + \alpha$) is the exact same for every cluster we evaluate (including a new one), we can ignore it during the loop. We just calculate the numerators (the "weights"), and then normalize everything at the very end by dividing by the total sum.So, the unnormalized prior weight for an existing cluster $k$ is simply m_counts[k].

So, our unnormalized priors (the first half of our probability equation) are simply the values in our m_counts array for existing clusters, and alpha for a new cluster.

In [ ]:
# --- 3. THE GIBBS SAMPLING LOOP ---
# We loop through every node to update its assignment
for i in range(N):
    # A. Remove node i's current influence from the tallies
    current_cluster = Z[i]
    m_counts[current_cluster] -= 1

    for j in range(N):
        if i == j:
            continue

        if X[i, j] == 1:
            n_counts[current_cluster, Z[j]] -= 1
        else:
            N_counts[current_cluster, Z[j]] -= 1

### 2. **Calculate the new probabilities**

Since this is a collapsed gibbs sampling implementation we will  completely mathematically integrated out the θ matrix (the block probabilities) using its Beta prior.

Because we collapsed those exact probabilities out of the model, we now use the Beta-Binomial relationship to calculate the probability "on the fly" using our running tallies and our hyperparameters (a and b).

Let's say we are testing what happens if we put node $i$ into cluster $k$. We look at its relationship with another node $j$, which is sitting in cluster $l$.If the actual network shows a link between them ($X_{i,j} = 1$), the probability of that link existing given the current state of the clusters is:$$\frac{n_{k,l} + a}{n_{k,l} + N_{k,l} + a + b}$$If there is no link between them ($X_{i,j} = 0$), the probability of that non-link is:$$\frac{N_{k,l} + b}{n_{k,l} + N_{k,l} + a + b}$$

To find the total likelihood of assigning node $i$ to cluster $k$, we have to evaluate these fractions for its connection to every other node $j$ in the network.

To avoid numerical overflow by calculating the products we use log probabilites: Because of the mathematical rule $\log(A \times B) = \log(A) + \log(B)$, we can swap our massive multiplication problem for a simple addition problem. Instead of multiplying the probabilities, we take the np.log() of each fraction and simply add them all up.

In [ ]:
# We are testing what happens if node i goes to cluster k
log_likelihood = 0.0

for j in range(N):
    if i == j:
        continue  # Skip comparing the node to itself

    l = Z[j]  # Find the cluster of the other node

    # The denominator is the same for both formulas
    denominator = n_counts[k, l] + N_counts[k, l] + a + b

    if X[i, j] == 1:
        # Formula for an existing link
        prob = (n_counts[k, l] + a) / denominator
    else:
        # Formula for no link
        prob = (N_counts[k, l] + b) / denominator

    # Add the log of the probability to our running total
    log_likelihood += np.log(prob)


NameError: name 'n_counts' is not defined

If we reverse the final log likelihood and take the exponential to reverse the effect np.exp(log_likelihood), it would result in underflow. Instead we take the log of the prior to maintain the corresponding values. So the

In [ ]:
prior = m_counts[k]
log_total_score = np.log(prior) + log_likelihood
log_total_score = log_likelihood_new + np.log(alpha)

We need to convert our log_scores back to standard numbers using np.exp(). But if we try to calculate np.exp(-450.2), Python will just output 0.0. All our probabilities would disappear.

To bypass this, we use a Log-Sum-Exp trick. It relies on the fact that we only care about the relative differences between these scores, not their absolute values. In standard math, if you divide all your weights by the same number, their relative proportions stay exactly the same. In log space, division is equivalent to subtraction.

In [ ]:
# log_scores = [-450.2, -455.1, -448.9]
max_log_score = np.max(log_scores)
safe_scores = log_scores - max_log_score
probabilities = np.exp(safe_scores)
# probabilities array is now [0.272, 0.002, 1.0]
# We need to normalize to sum to 1
probabilities = probabilities / np.sum(probabilities)

The function we need is np.random.choice(). We give it an array of options to choose from, and we give it your probabilities array using the p argument. It then rolls a weighted multi-sided die to make the selection. Since node $i$ can choose any of the $K$ existing clusters (indices 0 to K-1), or a brand new cluster (index K), our options are a list of numbers from 0 to K.

(Note: If new_cluster exactly equals our current $K$, it means the node chose the brand new cluster. If that happens, we would just increase $K$ by 1 and expand our $n$ and $N$ matrices to be $(K+1) \times (K+1)$).

In [ ]:
# Create an array of choices: [0, 1, ..., K]
cluster_options = np.arange(K + 1)

# Pick the new cluster weighted by our normalized probabilities
new_cluster = np.random.choice(cluster_options, p=probabilities)

# Update node i's assignment in our master list!
Z[i] = new_cluster

If node $i$ moves into a cluster that already exists, the code to add its stats back looks exactly like this:

(Note: If our network is undirected, meaning a link from $i$ to $j$ is the same as $j$ to $i$, we update both symmetrical sides of the matrix) However, we have to handle a special edge case if node $i$ chose a brand new cluster. If new_cluster is exactly equal to our current K, it means node $i$ is sitting at a new table. But our n_counts and N_counts matrices are currently only sized K by K. If we try to run the code above and access index K, Python will immediately throw an IndexError because that row and column don't exist ye

In [ ]:
m_counts[Z[i]] += 1

for j in range(N):
    if i == j:
        continue

    if X[i, j] == 1:
        n_counts[Z[i], Z[j]] += 1
        n_counts[Z[j], Z[i]] += 1  # (If the graph is undirected)
    else:
        N_counts[Z[i], Z[j]] += 1
        N_counts[Z[j], Z[i]] += 1  # (If the graph is undirected)

Because $N$ is the absolute ceiling, we can use pre-allocation. Instead of creating matrices of size $K \times K$ and trying to resize them on the fly, we initialize them at the very beginning to be their maximum possible size: $N \times N$.

In [ ]:
# Initialize to the maximum possible size
m_counts = np.zeros(N)
n_counts = np.zeros((N, N))
N_counts = np.zeros((N, N))